In [ ]:
!pip -q install pypdf sentence-transformers faiss-cpu google-genai

In [ ]:
from google.colab import files
uploaded = files.upload()
pdf_path = next(iter(uploaded))
print("Uploaded PDF:", pdf_path)

Saving Data Mining (S4103) Unit-wise Question Bank (1).pdf to Data Mining (S4103) Unit-wise Question Bank (1).pdf
Uploaded PDF: Data Mining (S4103) Unit-wise Question Bank (1).pdf


In [ ]:
from pypdf import PdfReader
reader = PdfReader(pdf_path)
text = ""
for page in reader.pages:
    text += (page.extract_text() or "") + "\n"
print(text[:2000])

Data Mining (S4103): Unit-wise Question
Bank (repeats merged)
JNTUH B.Tech IV Year I Sem, Data Mining (CSE). Compiled from 9 previous papers and
arranged as per the R18 syllabus.
Tags: 26M = Mar/Apr 2026, 25N = Oct/Nov 2025, 25M = Mar/Apr 2025, 24N = Oct/Nov
2024, 24M = Mar/Apr 2024, 23A = Apr/May 2023, 22N = Nov/Dec 2022, 22M = Mar/Apr
2022, 21N = Oct/Nov 2021.
★  = the same question was asked in 3 or more papers, so prepare it first.
UNIT I: Fundamentals, Data Processing, Data Warehouse, OLAP,
Sampling
SAQs
1. Define data mining. (24M)
2. Explain data mining as a step in the process of knowledge discovery. (23A)
3. ★  Define a data warehouse (data warehousing). (25N, 25M, 24N, 23A)
4. Distinguish between data mining and data warehousing. (26M)
5. List the advantages of a data warehouse. (22N)
6. What is a dimension table? (24M)
7. List the applications of data mining. (25M)
8. ★  What is sampling? Explain data sampling and mention two sampling techniques.
Illustrate stratified sampli

In [ ]:
def split_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0
    while start < len(text):
        chunk = text[start:start + chunk_size]
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks
chunks = split_text(text)
print("Total chunks:", len(chunks))
print("\nFirst chunk:\n", chunks[0])

Total chunks: 127

First chunk:
 Data Mining (S4103): Unit-wise Question
Bank (repeats merged)
JNTUH B.Tech IV Year I Sem, Data Mining (CSE). Compiled from 9 previous papers and
arranged as per the R18 syllabus.
Tags: 26M = Mar/Apr 2026, 25N = Oct/Nov 2025, 25M = Mar/Apr 2025, 24N = Oct/Nov
2024, 24M = Mar/Apr 2024, 23A = Apr/May 2023, 22N = Nov/Dec 2022, 22M = Mar/Apr
2022, 21N = Oct/Nov 2021.
★  = the same question was asked in 3 or more papers, so prepare it first.
UNIT I: Fundamentals, Data Processing, Data Warehouse, OLAP,
Sampling
SAQs
1. Define data mining. (24M)
2. Explain data mining as a step in the process of knowledge discovery. (23A)
3. ★  Define a data warehouse (data warehousing). (25N, 25M, 24N, 23A)
4. Distinguish between data mining and data warehousing. (26M)
5. List the advantages of a data warehouse. 


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True
)
embeddings = embeddings.astype("float32")
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
print("FAISS database created!")
print("Number of stored chunks:", index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS database created!
Number of stored chunks: 127


In [ ]:
from google import genai
from getpass import getpass
api_key = getpass("Enter your Gemini API key: ")
client = genai.Client(api_key=api_key)
print("Gemini initialized. No API request sent yet.")

Enter your Gemini API key: ··········
Gemini initialized. No API request sent yet.


In [ ]:
def ask_pdf(question):
    if not question.strip():
        print("Please enter a question.")
        return
    # Local embedding: no Gemini request
    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    ).astype("float32")
    # Local FAISS search: no Gemini request
    k = min(3, index.ntotal)
    if k == 0:
        print("No PDF text is available.")
        return
    scores, indices = index.search(question_embedding, k)
    context = "\n\n".join(
        chunks[i] for i in indices[0] if i >= 0
    )
    prompt = f"""
You are a PDF question-answering assistant.
Answer using only the provided PDF context.
If the context does not contain the answer, say:
"The answer was not found in the provided PDF."
Keep the answer clear and concise.

PDF CONTEXT:
{context}

QUESTION:
{question}
"""
    # Exactly one Gemini generation request
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    print("\nAnswer:", response.text)

In [ ]:
while True:
    question = input("\nAsk a question (or type 'exit'): ")
    if question.strip().lower() == "exit":
        print("Chat ended.")
        break
    if question.strip():
        try:
            ask_pdf(question)
        except Exception as e:
            print("Request failed:", e)


Ask a question (or type 'exit'): exit
Chat ended.


In [ ]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
# 1. Extract PDF text with page numbers
reader = PdfReader(pdf_path)
pages = []
for page_num, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text() or ""
    if page_text.strip():
        pages.append({
            "page": page_num,
            "text": page_text
        })
if not pages:
    raise ValueError("No extractable text found in this PDF. Scanned PDFs need OCR.")
# 2. Split text into overlapping chunks
chunks = []
chunk_size = 800
overlap = 150
for page in pages:
    page_text = page["text"]
    for start in range(0, len(page_text), chunk_size - overlap):
        chunk_text = page_text[start:start + chunk_size]
        if chunk_text.strip():
            chunks.append({
                "text": chunk_text,
                "page": page["page"]
            })
# 3. Generate local embeddings (no Gemini requests)
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
chunk_texts = [chunk["text"] for chunk in chunks]
embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")
# 4. Create FAISS index
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print("PDF processed successfully!")
print("Pages indexed:", len(pages))
print("Chunks indexed:", len(chunks))
print("Gemini requests used during indexing: 0")
# 5. Ask questions with page references
def ask_pdf(question):
    if not question.strip():
        print("Please enter a question.")
        return
    # Local embedding and retrieval
    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    ).astype("float32")
    k = min(3, index.ntotal)
    scores, indices = index.search(query_embedding, k)
    retrieved = [
        chunks[i]
        for i in indices[0]
        if i >= 0
    ]
    context = "\n\n".join(
        f"[PDF Page {chunk['page']}]\n{chunk['text']}"
        for chunk in retrieved
    )
    prompt = f"""
You are a PDF question-answering assistant.
Answer using only the provided context.
Cite the PDF page numbers supporting your answer.
If the answer is absent from the context, say:
"The answer was not found in the provided PDF."

Context:
{context}

Question: {question}
"""
    # One Gemini generation request per question
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    print("\nAnswer:")
    print(response.text)
    source_pages = sorted(set(
        chunk["page"] for chunk in retrieved
    ))
    print("\nRetrieved PDF pages:", source_pages)
# 6. Interactive chatbot
print("\nPDF Chatbot Ready!")
print("Type 'exit' to stop.")
while True:
    question = input("\nYour question: ")
    if question.strip().lower() == "exit":
        print("Chatbot closed.")
        break
    if question.strip():
        try:
            ask_pdf(question)
        except Exception as error:
            print("Error:", error)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

PDF processed successfully!
Pages indexed: 41
Chunks indexed: 149
Gemini requests used during indexing: 0

PDF Chatbot Ready!
Type 'exit' to stop.

Your question: exit
Chatbot closed.


In [ ]:

import streamlit as st
import io
import time
import re
import numpy as np
import faiss

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from google import genai


# ---------------- PAGE CONFIGURATION ----------------

st.set_page_config(
    page_title="RAG PDF Assistant",
    page_icon="📚",
    layout="wide"
)

st.title("📚 RAG-Based PDF Question Answering System")
st.write("Upload a PDF and ask questions based on its content.")


# ---------------- GEMINI API ----------------

st.sidebar.header("Gemini API Settings")

api_key = st.sidebar.text_input(
    "Enter Gemini API Key",
    type="password"
)

MODEL_NAME = "gemini-3.8-flash"


# ---------------- LOAD EMBEDDING MODEL ----------------

@st.cache_resource
def load_embedding_model():
    return SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


with st.spinner("Loading embedding model..."):
    embedding_model = load_embedding_model()


# ---------------- TEXT EXTRACTION ----------------

def extract_pdf_text(pdf_bytes):

    reader = PdfReader(io.BytesIO(pdf_bytes))

    pages = []

    for page_number, page in enumerate(reader.pages, start=1):

        page_text = page.extract_text() or ""

        if page_text.strip():
            pages.append({
                "page": page_number,
                "text": page_text
            })

    return pages


# ---------------- TEXT CHUNKING ----------------

def create_chunks(pages, chunk_size=1000, overlap=180):

    chunks = []

    for page in pages:

        text = re.sub(r"\s+", " ", page["text"]).strip()

        start = 0

        while start < len(text):

            end = min(start + chunk_size, len(text))

            chunk_text = text[start:end]

            if chunk_text.strip():
                chunks.append({
                    "text": chunk_text,
                    "page": page["page"]
                })

            if end == len(text):
                break

            start = end - overlap

    return chunks


# ---------------- CREATE FAISS INDEX ----------------

def create_faiss_index(chunks):

    texts = [chunk["text"] for chunk in chunks]

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    embeddings = np.asarray(embeddings, dtype="float32")

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(dimension)

    index.add(embeddings)

    return index


# ---------------- RETRIEVE RELEVANT CHUNKS ----------------

def retrieve_chunks(question, chunks, index, top_k=6):

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    question_embedding = np.asarray(
        question_embedding,
        dtype="float32"
    )

    k = min(top_k, len(chunks))

    scores, indices = index.search(question_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx >= 0:
            results.append({
                "text": chunks[idx]["text"],
                "page": chunks[idx]["page"],
                "score": float(score)
            })

    return results


# ---------------- GEMINI ANSWER GENERATION ----------------

def generate_answer(client, question, retrieved_chunks):

    context = "\n\n".join([
        f"[Page {chunk['page']}]\n{chunk['text']}"
        for chunk in retrieved_chunks
    ])

    prompt = f"""
You are a helpful PDF question-answering assistant.

Answer the user's question using ONLY the information
provided in the PDF context below.

Instructions:
1. Give a clear and accurate answer.
2. Use simple language.
3. Include relevant details from the PDF.
4. Mention page numbers when possible.
5. Do not invent information.
6. If the answer is not available in the PDF, clearly say:
   "The answer is not available in the uploaded PDF."

PDF CONTEXT:
{context}

USER QUESTION:
{question}

ANSWER:
"""

    # First Gemini request
    try:

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )

        return response.text or "No answer was generated."

    except Exception as first_error:

        error_message = str(first_error)

        # Retry only for temporary server errors
        if "503" in error_message or "UNAVAILABLE" in error_message:

            time.sleep(2)

            try:

                response = client.models.generate_content(
                    model=MODEL_NAME,
                    contents=prompt
                )

                return response.text or "No answer was generated."

            except Exception:

                return (
                    "Gemini is temporarily experiencing high demand. "
                    "Please try again in a little while."
                )

        return f"Gemini API Error: {error_message}"


# ---------------- PDF UPLOAD ----------------

uploaded_file = st.file_uploader(
    "Upload your PDF document",
    type=["pdf"]
)


if uploaded_file is not None:

    pdf_bytes = uploaded_file.getvalue()

    if not pdf_bytes.startswith(b"%PDF-"):

        st.error("This file is not a valid PDF. Please upload an actual PDF.")

    else:

        if st.button("Process PDF"):

            try:

                with st.spinner("Extracting PDF text..."):

                    pages = extract_pdf_text(pdf_bytes)

                if not pages:

                    st.error(
                        "No readable text found. This may be a scanned PDF. "
                        "OCR is required."
                    )

                    st.stop()

                with st.spinner("Creating text chunks..."):

                    chunks = create_chunks(pages)

                if not chunks:

                    st.error("No text chunks could be created.")
                    st.stop()

                with st.spinner("Creating embeddings and FAISS index..."):

                    index = create_faiss_index(chunks)

                st.session_state["chunks"] = chunks
                st.session_state["index"] = index
                st.session_state["pdf_name"] = uploaded_file.name

                st.success("PDF processed successfully!")

                st.write("**Document:**", uploaded_file.name)
                st.write("**Pages extracted:**", len(pages))
                st.write("**Chunks created:**", len(chunks))

            except Exception as e:

                st.error(f"PDF processing error: {str(e)}")


# ---------------- QUESTION ANSWERING ----------------

if "chunks" in st.session_state and "index" in st.session_state:

    st.divider()

    st.subheader("💬 Ask Questions About Your PDF")

    question = st.chat_input("Enter your question here...")

    if question:

        st.chat_message("user").write(question)

        if not api_key:

            st.warning("Please enter your Gemini API key in the sidebar.")

        else:

            try:

                client = genai.Client(api_key=api_key)

                with st.spinner("Searching PDF and generating answer..."):

                    retrieved_chunks = retrieve_chunks(
                        question,
                        st.session_state["chunks"],
                        st.session_state["index"]
                    )

                    answer = generate_answer(
                        client,
                        question,
                        retrieved_chunks
                    )

                st.chat_message("assistant").write(answer)

                with st.expander("📄 View Retrieved PDF Sources"):

                    for i, chunk in enumerate(retrieved_chunks, start=1):

                        st.markdown(
                            f"**Source {i} — Page {chunk['page']}**"
                        )

                        st.write(chunk["text"])

                        st.caption(
                            f"Similarity score: {chunk['score']:.3f}"
                        )

                        st.divider()

            except Exception as e:

                st.error(f"Error: {str(e)}")


# ---------------- SIDEBAR INFORMATION ----------------

st.sidebar.divider()

st.sidebar.subheader("About This Project")

st.sidebar.write("""
**RAG PDF Assistant**

Technologies:
- Python
- Streamlit
- Google Gemini API
- Sentence Transformers
- FAISS
- PyPDF

Features:
- PDF text extraction
- Text chunking
- Semantic search
- Context-based answers
- Source page references
""")


2026-10-31 15:11:32.756 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-31 15:11:32.758 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-31 15:11:33.341 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-10-31 15:11:33.344 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-31 15:11:33.357 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-31 15:11:33.363 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-31 15:11:33.371 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-10-31 15:11:35.521 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-31 15:11:35.526 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-31 15:11:35.528 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-31 15:11:35.531 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-31 15:11:35.535 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-31 15:11:35.545 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-31 15:11:35.550 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-31 15:11:35.555 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [ ]:
!pip -q install streamlit pyngrok

In [ ]:
from getpass import getpass
from pyngrok import ngrok
import subprocess
import time
import requests

# Enter ngrok authentication token
ngrok_token = getpass("Enter your ngrok auth token: ")
ngrok.set_auth_token(ngrok_token)

# Stop any previously running Streamlit processes
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)

# Start Streamlit
process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=open("/content/streamlit.log", "w"),
    stderr=subprocess.STDOUT
)

# Wait for Streamlit to start
time.sleep(5)

# Create public ngrok tunnel
public_url = ngrok.connect(8501, "http")

print("✅ Your RAG PDF Assistant is running!")
print("🌐 Open your application:", public_url)

Enter your ngrok auth token: ··········
✅ Your RAG PDF Assistant is running!
🌐 Open your application: NgrokTunnel: "https://spiffy-stubble-pasta.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
# Update the Gemini model in app.py
from pathlib import Path
path = Path("app.py")
code = path.read_text()
code = code.replace(
    "gemini-2.5-flash",
    "gemini-3.8-flash"
)
path.write_text(code)
print("Updated Gemini model successfully!")

Updated Gemini model successfully!


In [ ]:
import subprocess
import time
from pyngrok import ngrok
# Stop the previous Streamlit process
try:
    process.terminate()
    process.wait(timeout=5)
except Exception:
    pass
# Close the old tunnel
ngrok.kill()
# Start Streamlit again
process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)
time.sleep(3)
public_url = ngrok.connect(8501)
print("Open your updated app:", public_url)

Open your updated app: NgrokTunnel: "https://spiffy-stubble-pasta.ngrok-free.dev" -> "http://localhost:8501"
